In [ ]:
import pandas as pd


#data from the weather stations
headers = ['yyyy', 'mm', 'tmax', 'tmin', 'af', 'rain', 'sun']
dtype = {'yyyy': int, 'mm':int, 'tmax degC':float, 'tmin degC':float, 'af days':int, 'rain mm':float, 'sun hours':float}
weather_station_df = pd.read_csv(r"suttonboningtondata.txt", sep=r'\s+', header=None, on_bad_lines='skip')
weather_station_df.columns = headers

#print(df.head())


headers = ['Unix', 'Aggregate', 'Appliance1', 'Appliance2', 'Appliance3', 'Appliance4', 'Appliance5', 'Appliance6', 'Appliance7', 'Appliance8', 'Appliance9']

df1 = pd.read_csv(r"House1.csv")
df2 = pd.read_csv(r"House2.csv")
df3 = pd.read_csv(r"House3.csv")
df4 = pd.read_csv(r"House4.csv")
df5 = pd.read_csv(r"House5.csv")
df6 = pd.read_csv(r"House6.csv")
df7 = pd.read_csv(r"House7.csv")
df8 = pd.read_csv(r"House8.csv")
df9 = pd.read_csv(r"House9.csv")
df10 = pd.read_csv(r"House10.csv")
df11 = pd.read_csv(r"House11.csv")
df12 = pd.read_csv(r"House12.csv")
df13 = pd.read_csv(r"House13.csv")
#No data for house 14
df15 = pd.read_csv(r"House15.csv")
df16 = pd.read_csv(r"House16.csv")
df17 = pd.read_csv(r"House17.csv")
df18 = pd.read_csv(r"House18.csv")
df19 = pd.read_csv(r"House19.csv")
df20 = pd.read_csv(r"House20.csv")
df21 = pd.read_csv(r"House21.csv")

dataframes = [df1, df2, df3, df4, df5, df6, df7, df8, df9, df10,
              df11, df12, df13, df15, df16, df17, df18, df19, df20, df21]

#add a header to the REFIT data
for data in dataframes:
    data.columns = headers

#Put which house the data is from for each house
for i, data in enumerate(dataframes):
    if i < 13:
        data['House'] = i + 1
    else: 
        data['House'] = i + 2


#Convert the Unix time to mm/dd/yyyy
#Group the data by the day and average out the values
condensed_data = []

for data in dataframes:
    data['datetime'] = pd.to_datetime(data['Unix'], unit='s')
    data['date'] = data['datetime'].dt.date
    condensed = data.groupby('date').agg({'Aggregate':'mean', 'Appliance1':'mean', 'Appliance2':'mean', 'Appliance3':'mean', 'Appliance4':'mean', 'Appliance5':'mean', 'Appliance6':'mean', 'Appliance7':'mean', 'Appliance8':'mean', 'Appliance9':'mean', 'House':'mean'}).reset_index()
    condensed_data.append(condensed)

total_REFIT_data = pd.concat(condensed_data, ignore_index=True)

#Data from visualcrossing
visualcrossing_df = pd.read_csv(r"Loughborough 2014-01-01 to 2015-12-31.csv")
visualcrossing_df.drop(columns=['name', 'description', 'icon', 'stations'], inplace=True)


In [ ]:
from tabulate import tabulate

temp_REFIT = total_REFIT_data.copy()
temp_visual = visualcrossing_df.copy()

#sorting data by date
start_date = '2014-01-01'
end_date = '2015-12-31'
start_date = pd.to_datetime(start_date)
end_date = pd.to_datetime(end_date)

temp_REFIT['date'] = pd.to_datetime(temp_REFIT['date'])
filtered_REFIT = temp_REFIT[(temp_REFIT['date'] >= start_date) & (temp_REFIT['date'] <= end_date)]
#extracting year and month
filtered_REFIT['year'] = filtered_REFIT['date'].dt.year
filtered_REFIT['month'] = filtered_REFIT['date'].dt.month

#grouping by the month
grouped = filtered_REFIT.groupby(['year', 'month'])

monthly_REFIT = {}

for (year, month), data in grouped:
    month_str = f"{year}-{month:02}"
    monthly_REFIT[month_str] = data.copy()


#doing the same thing for the visualcrossing dataframe
temp_visual['date'] = pd.to_datetime(temp_visual['datetime'])

filtered_visual = temp_visual[(temp_visual['date'] >= start_date) & (temp_visual['date'] <= end_date)]
filtered_visual['year'] = filtered_visual['date'].dt.year
filtered_visual['month'] = filtered_visual['date'].dt.month

grouped = filtered_visual.groupby(['year', 'month'])

monthy_visual = {}

for (year, month), data in grouped:
    month_str = f"{year}-{month:02}"
    monthy_visual[month_str] = data.copy()

#combine the dataframes into one
monthly_electricty_and_weather = {}
for month_str, refit_df in monthly_REFIT.items():
    if month_str in monthy_visual:
        visual_df = monthy_visual[month_str]
        combined = pd.merge(refit_df, visual_df, on='date', how='inner')
        monthly_electricty_and_weather[month_str] = combined
    else:
        print(f"\n\nMissing weather data for {month_str}\n\n")


print(monthly_electricty_and_weather['2014-01'].columns)
print(tabulate(monthly_electricty_and_weather, headers='keys', tablefmt='fancy_grid', showindex=False))

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for date in monthly_electricty_and_weather:
    x = monthly_electricty_and_weather[date]['precip']
    y = monthly_electricty_and_weather[date]['Aggregate']
    hue = monthly_electricty_and_weather[date]['House']
    sns.scatterplot(x = x,
                    y = y,
                    hue = hue,
                    palette = 'tab20',
                    legend = False,
                    data = monthly_electricty_and_weather[date])
    #plt.xlim(32,70)
    plt.ylim(0, 1500)
    plt.title(f'Average Power Usage vs Temperatue for {date}')
    plt.show()

In [ ]:
'''
EDIT THE PLOTS AS NEEDED
'''

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

for date in monthly_electricty_and_weather:
    x = monthly_electricty_and_weather[date]['humidity']
    y = monthly_electricty_and_weather[date]['Aggregate']
    hue = monthly_electricty_and_weather[date]['House']
    sns.scatterplot(x = x,
                    y = y,
                    hue = hue,
                    palette = 'tab20',
                    legend = False,
                    data = monthly_electricty_and_weather[date])
    plt.ylim(0, 1500)
    plt.xlim(60, 100)
    plt.title(f'Average Power Usage vs Humidity for {date}')
    plt.show()

In [ ]:
#Looking at the data based off the year
#going to look at 2014 year because it has a full year of data
temp_REFIT = total_REFIT_data.copy()
temp_visual = visualcrossing_df.copy()

#sorting data by date
start_date = '2014-01-01'
end_date = '2014-12-31'
start_date = pd.to_datetime(start_date)
end_date = pd.to_datetime(end_date)

#converting to date time and limiting to just 2014
temp_REFIT['date'] = pd.to_datetime(temp_REFIT['date'])
filtered_REFIT = temp_REFIT[(temp_REFIT['date'] >= start_date) & (temp_REFIT['date'] <= end_date)]
filtered_REFIT = filtered_REFIT.sort_values(by='date')

#doing the same thing for the visualcrossing dataframe
temp_visual['date'] = pd.to_datetime(temp_visual['datetime'])
filtered_visual = temp_visual[(temp_visual['date'] >= start_date) & (temp_visual['date'] <= end_date)]
filtered_visual = filtered_visual.sort_values(by='date')

#combine the dataframes into one
yearly_electricity_and_weather = pd.merge(filtered_REFIT, filtered_visual, on ='date', how='inner')

print(yearly_electricity_and_weather.columns)

In [ ]:
import datetime
#seperating the data into weekdays and weekends
weekdays_df = yearly_electricity_and_weather[yearly_electricity_and_weather['date'].dt.weekday < 5].copy()
weekends_df = yearly_electricity_and_weather[yearly_electricity_and_weather['date'].dt.weekday >= 5].copy()

In [ ]:
#Histogram of both the weekend and weedays
sns.histplot(data = weekdays_df, x = weekdays_df['Aggregate'])
plt.xlim(0, 2000)
plt.ylim(0, 350)
plt.title('Histogram for Weekday Usage')
plt.show()

sns.histplot(data = weekends_df, x = weekends_df['Aggregate'])
plt.xlim(0, 2000)
plt.ylim(0, 350)
plt.title('Histogram for Weekend Usage')
plt.show()

In [ ]:
#remove the outliers to try and clean up the data
import numpy as np
from scipy import stats

z_scores = np.abs(stats.zscore(yearly_electricity_and_weather['Aggregate']))
filter_yearly_electricity_and_weather = yearly_electricity_and_weather[z_scores < 3]

In [ ]:
y = filter_yearly_electricity_and_weather['precip']
x = filter_yearly_electricity_and_weather['date']
hue = filter_yearly_electricity_and_weather['Aggregate']
sns.scatterplot(x = x,
                    y = y,
                    hue = hue,
                    data = filter_yearly_electricity_and_weather)
plt.title(f'Average Power Usage for 2014')

In [ ]:
y = filter_yearly_electricity_and_weather['uvindex']
x = filter_yearly_electricity_and_weather['date']
hue = filter_yearly_electricity_and_weather['Aggregate']
sns.scatterplot(x = x,
                    y = y,
                    hue = hue,
                    palette = 'inferno',
                    data = filter_yearly_electricity_and_weather)
plt.title(f'Average Power Usage for 2014')

In [ ]:
import matplotlib.ticker as ticker
from sklearn.linear_model import LinearRegression

x = filter_yearly_electricity_and_weather['humidity']
y = filter_yearly_electricity_and_weather['Aggregate']
sns.barplot(x = x,
            y = y,
            data = filter_yearly_electricity_and_weather)
plt.title('Bar Plot of Humidity vs Aggregate')
plt.gca().xaxis.set_major_locator(ticker.MaxNLocator(nbins = 5))
plt.show()

In [ ]:
'''
Linear Regression doesn't seem to really match well with the data. Lots of outliers. So I chose not to use this
'''
x = filter_yearly_electricity_and_weather['humidity']
y = filter_yearly_electricity_and_weather['Aggregate']
plt.figure(figsize=(10, 6))  # Adjust figure size
sns.regplot(x=x, 
            y=y, 
            data=filter_yearly_electricity_and_weather,
            robust = True,
            scatter_kws={'s': 50, 'alpha': 0.9, 'color': 'lavender'}, #scatter customization
            line_kws={'color': 'red', 'linewidth': 2}, #line customization
            ci=95)  # Add 95% confidence intervals

plt.title('Linear Regression Plot', fontsize=16)
plt.xlabel('Humidity', fontsize=14)
plt.ylabel('Aggregate', fontsize=14)
plt.grid(True, linestyle='--', alpha=0.6) #add grid.
plt.show()

Starting to look at which variables most affect the aggregate power using the models. Going to use Temp, Humidity, Precipitation, and the Day of the Week

In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import matplotlib.pyplot as plt
import numpy as np

#creating a dataset that the model will look at
final_df = yearly_electricity_and_weather[['date', 'Aggregate', 'House', 'temp', 'humidity', 'precip']]

#create weekend or weekday column
final_df['day_of_week'] = final_df['date'].dt.day_of_week
final_df['is_weekend'] = final_df['day_of_week'].apply(lambda x: 1 if x >= 5 else 0)
final_df = final_df.drop('day_of_week', axis=1)


In [84]:
#create the df for the model
df_model = final_df.drop('date', axis=1)
df_model['temp'] = df_model['temp'].astype(float)
df_model['humidity'] = df_model['humidity'].astype(float)
df_model['precip'] = df_model['precip'].astype(float)
df_model['Aggregate'] = df_model['Aggregate'].astype(float)
df_model['is_weekend'] = df_model['is_weekend'].astype(int)

#training model
y = df_model['Aggregate']
features = ['temp', 'humidity', 'precip', 'is_weekend']
X = df_model[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#start RF
rf_model = RandomForestRegressor(n_estimators=500, random_state=42)
rf_model.fit(X_train, y_train)
predictions = rf_model.predict(X_test)

#Calculate Error
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
print(f'Mean Square Error: {mse}')
print(f'Root Square Mean Error: {rmse}')

#Determine important features
feature_importance = rf_model.feature_importances_
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importance})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

#Display results
print(importance_df)

Mean Square Error: 85653.8833710058
Root Square Mean Error: 292.6668470650644
      Feature  Importance
0        temp    0.464304
1    humidity    0.293776
2      precip    0.191010
3  is_weekend    0.050909


Very high error so redfining the dataset to try and lower the error. Going to combine the houses and average out the values see if that helps with the error


In [85]:
df_model = final_df.drop('House', axis=1)

#combining each house into one and averaging out the values
df_model = df_model.groupby('date').agg(
    Aggregate = ('Aggregate', 'mean'),
    Temp = ('temp', 'mean'),
    Humid = ('humidity', 'mean'),
    Precip = ('precip', 'mean'),
    is_weekend = ('is_weekend', 'first')
).reset_index()

df_model['Aggregate'] = df_model['Aggregate'].astype(float)
df_model['Temp'] = df_model['Temp'].astype(float)
df_model['Humid'] = df_model['Humid'].astype(float)
df_model['is_weekend'] = df_model['is_weekend'].astype(int)
df_model['Precip'] = df_model['Precip'].astype(float)
df_model = df_model.drop('date', axis=1)

#training model
y = df_model['Aggregate']
features = ['Temp', 'Humid', 'Precip', 'is_weekend']
X = df_model[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#start RF
rf_model = RandomForestRegressor(n_estimators=300, random_state=42)
rf_model.fit(X_train, y_train)
predictions = rf_model.predict(X_test)

#Calculate Error
mse = mean_squared_error(y_test, predictions)
rmse = np.sqrt(mse)
print(f'Mean Square Error: {mse}')
print(f'Root Square Mean Error: {rmse}')

#Determine important features
feature_importance = rf_model.feature_importances_
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importance})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

#Display results
print(importance_df)

Mean Square Error: 5300.595583614457
Root Square Mean Error: 72.80518926295335
      Feature  Importance
0        Temp    0.621924
1       Humid    0.190767
2      Precip    0.138028
3  is_weekend    0.049281


Error was still high now going to look at each individual house

In [86]:
df_model = final_df.drop('date', axis=1)
df_model['temp'] = df_model['temp'].astype(float)
df_model['humidity'] = df_model['humidity'].astype(float)
df_model['precip'] = df_model['precip'].astype(float)
df_model['Aggregate'] = df_model['Aggregate'].astype(float)
df_model['is_weekend'] = df_model['is_weekend'].astype(int)

df_model = df_model.sort_values('House', ascending=True)

#creating a serperate dataframe for each house
house_dataframes = {}
for house in df_model['House'].unique():
    temp_df = df_model[df_model['House'] == house].copy()
    house_dataframes[house] = temp_df
'''
for id, data in house_dataframes.items():
    print(f'House {id}, entries {len(data)}')
'''

for id, data in house_dataframes.items():
    #training model
    y = data['Aggregate']
    features = ['temp', 'humidity', 'precip', 'is_weekend']
    X = data[features]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    #start RF
    rf_model = RandomForestRegressor(n_estimators=100, random_state=42)
    rf_model.fit(X_train, y_train)
    predictions = rf_model.predict(X_test)

    #Calculate Error
    mse = mean_squared_error(y_test, predictions)
    rmse = np.sqrt(mse)
    print(f'Mean Square Error for House {id}: {mse}')
    print(f'Root Square Mean Error for House {id}: {rmse}')

    #Determine important features
    feature_importance = rf_model.feature_importances_
    importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importance})
    importance_df = importance_df.sort_values(by='Importance', ascending=False)

    #Display results
    print(f'Importance for House {id}: \n{importance_df}\n')


Mean Square Error for House 1.0: 29184.26496417677
Root Square Mean Error for House 1.0: 170.83402753601746
Importance for House 1.0: 
      Feature  Importance
0        temp    0.542146
1    humidity    0.256020
2      precip    0.176309
3  is_weekend    0.025525

Mean Square Error for House 2.0: 32314.07437610379
Root Square Mean Error for House 2.0: 179.76115925333755
Importance for House 2.0: 
      Feature  Importance
0        temp    0.454093
1    humidity    0.307714
2      precip    0.186964
3  is_weekend    0.051229

Mean Square Error for House 3.0: 42868.625605299734
Root Square Mean Error for House 3.0: 207.04739941689616
Importance for House 3.0: 
      Feature  Importance
0        temp    0.473632
1    humidity    0.285901
2      precip    0.193703
3  is_weekend    0.046765

Mean Square Error for House 4.0: 6513.458463323166
Root Square Mean Error for House 4.0: 80.70600016927592
Importance for House 4.0: 
      Feature  Importance
0        temp    0.381125
1    humidity  

RMSE is still very high going to try looking using hyperparaters with RandomCV

In [87]:
from sklearn.model_selection import RandomizedSearchCV
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_squared_error
import numpy as np
from scipy.stats import randint, uniform

#This one is for average all the houses into 1
df_model = final_df.drop('House', axis=1)

df_model = df_model.groupby('date').agg(
    Aggregate = ('Aggregate', 'mean'),
    Temp = ('temp', 'mean'),
    Humid = ('humidity', 'mean'),
    Precip = ('precip', 'mean'),
    is_weekend = ('is_weekend', 'first')
).reset_index()

df_model['Aggregate'] = df_model['Aggregate'].astype(float)
df_model['Temp'] = df_model['Temp'].astype(float)
df_model['Humid'] = df_model['Humid'].astype(float)
df_model['is_weekend'] = df_model['is_weekend'].astype(int)
df_model['Precip'] = df_model['Precip'].astype(float)
df_model = df_model.drop('date', axis=1)


#training model
y = df_model['Aggregate']
features = ['Temp', 'Humid', 'Precip', 'is_weekend']
X = df_model[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Random numbers for the paramaters just trying to see which options work best
param_distr = {
    'n_estimators': randint(100,300),
    'max_depth': [None] + list(randint(5,50).rvs(10)),
    'min_samples_split': randint(2,20),
    'min_samples_leaf': randint(1,10),
    'max_features': ['sqrt', 'log2']
}
#start RF
rf_model = RandomForestRegressor(random_state=42)

#Start the RandomSearchCV
random_search = RandomizedSearchCV(estimator=rf_model,
                                   param_distributions=param_distr,
                                   n_iter=20,
                                   cv=5,
                                   random_state=42)
random_search.fit(X_train, y_train)

#Print which is the best options
print(f'Best paramaters: {random_search.best_params_}')
print(f'Best RMSE: {random_search.best_score_}')
print(f'Best estimator: {random_search.best_estimator_}')

feature_importance = random_search.best_estimator_.feature_importances_
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importance})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

#Display results
print(importance_df)


Best paramaters: {'max_depth': 48, 'max_features': 'log2', 'min_samples_leaf': 4, 'min_samples_split': 15, 'n_estimators': 194}
Best RMSE: 0.4005903193059419
Best estimator: RandomForestRegressor(max_depth=48, max_features='log2', min_samples_leaf=4,
                      min_samples_split=15, n_estimators=194, random_state=42)
      Feature  Importance
0        Temp    0.656292
1       Humid    0.203899
2      Precip    0.087605
3  is_weekend    0.052203


Using the hyperparameters was kinda better. Model is 40% accurate, which I guess isn't bad. Weather isn't the only predicitor for power consumption, so it shouldn't be able to 100% predict the values based off weather patterns.

Next going to look at DecisionTree see how affective it is

In [88]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error
import numpy as np

#create the df for the model
df_model = final_df.drop('date', axis=1)
df_model['temp'] = df_model['temp'].astype(float)
df_model['humidity'] = df_model['humidity'].astype(float)
df_model['precip'] = df_model['precip'].astype(float)
df_model['Aggregate'] = df_model['Aggregate'].astype(float)
df_model['is_weekend'] = df_model['is_weekend'].astype(int)

#training model
y = df_model['Aggregate']
features = ['temp', 'humidity', 'precip', 'is_weekend']
X = df_model[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Start Decision Tree
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
dt_predict = dt_model.predict(X_test)

#calculate error
rmse = np.sqrt(mean_squared_error(y_test, dt_predict))
print(f'RMSE: {rmse}')

#find important features
feature_importance = dt_model.feature_importances_
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importance})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

#Display results
print(importance_df)

RMSE: 292.6358630318945
      Feature  Importance
0        temp    0.578782
1    humidity    0.233358
2      precip    0.132608
3  is_weekend    0.055252


Going to repeat the same process, looking at the normal dataset, then combining the houses, then looking at each individual house, then looking at it with hyper parameters.
Next one is combining the houses

In [90]:
df_model = final_df.drop('House', axis=1)

#combining each house into one and averaging out the values
df_model = df_model.groupby('date').agg(
    Aggregate = ('Aggregate', 'mean'),
    Temp = ('temp', 'mean'),
    Humid = ('humidity', 'mean'),
    Precip = ('precip', 'mean'),
    is_weekend = ('is_weekend', 'first')
).reset_index()

df_model['Aggregate'] = df_model['Aggregate'].astype(float)
df_model['Temp'] = df_model['Temp'].astype(float)
df_model['Humid'] = df_model['Humid'].astype(float)
df_model['is_weekend'] = df_model['is_weekend'].astype(int)
df_model['Precip'] = df_model['Precip'].astype(float)
df_model = df_model.drop('date', axis=1)

#training model
y = df_model['Aggregate']
features = ['Temp', 'Humid', 'Precip', 'is_weekend']
X = df_model[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#Start Decision Tree
dt_model = DecisionTreeRegressor(random_state=42)
dt_model.fit(X_train, y_train)
dt_predict = dt_model.predict(X_test)

#calculate error
rmse = np.sqrt(mean_squared_error(y_test, dt_predict))
print(f'RMSE: {rmse}')

#find important features
feature_importance = dt_model.feature_importances_
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importance})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

#Display results
print(importance_df)

RMSE: 100.021218851476
      Feature  Importance
0        Temp    0.610473
1       Humid    0.211782
2      Precip    0.108667
3  is_weekend    0.069079


Similar results to the RandomForest model

In [91]:
df_model = final_df.drop('date', axis=1)
df_model['temp'] = df_model['temp'].astype(float)
df_model['humidity'] = df_model['humidity'].astype(float)
df_model['precip'] = df_model['precip'].astype(float)
df_model['Aggregate'] = df_model['Aggregate'].astype(float)
df_model['is_weekend'] = df_model['is_weekend'].astype(int)

df_model = df_model.sort_values('House', ascending=True)

#creating a serperate dataframe for each house
house_dataframes = {}
for house in df_model['House'].unique():
    temp_df = df_model[df_model['House'] == house].copy()
    house_dataframes[house] = temp_df
'''
for id, data in house_dataframes.items():
    print(f'House {id}, entries {len(data)}')
'''

for id, data in house_dataframes.items():
    #training model
    y = data['Aggregate']
    features = ['temp', 'humidity', 'precip', 'is_weekend']
    X = data[features]
    X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

    #Start Decision Tree
    dt_model = DecisionTreeRegressor(random_state=42)
    dt_model.fit(X_train, y_train)
    dt_predict = dt_model.predict(X_test)

    #calculate error
    rmse = np.sqrt(mean_squared_error(y_test, dt_predict))
    print(f'RMSE for House {id}: {rmse}')

    #find important features
    feature_importance = dt_model.feature_importances_
    importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importance})
    importance_df = importance_df.sort_values(by='Importance', ascending=False)

    #Display results
    print(f'Features for House {id}: \n{importance_df}\n')

RMSE for House 1.0: 239.10288376926994
Features for House 1.0: 
      Feature  Importance
0        temp    0.436733
1    humidity    0.345348
2      precip    0.205510
3  is_weekend    0.012409

RMSE for House 2.0: 212.27331676379174
Features for House 2.0: 
      Feature  Importance
0        temp    0.455999
1    humidity    0.286617
2      precip    0.211772
3  is_weekend    0.045612

RMSE for House 3.0: 308.35854005024555
Features for House 3.0: 
      Feature  Importance
0        temp    0.540472
1    humidity    0.307192
2      precip    0.121401
3  is_weekend    0.030935

RMSE for House 4.0: 103.39777006618739
Features for House 4.0: 
      Feature  Importance
0        temp    0.403798
1    humidity    0.365447
2      precip    0.209273
3  is_weekend    0.021482

RMSE for House 5.0: 355.6409602685717
Features for House 5.0: 
      Feature  Importance
1    humidity    0.368103
0        temp    0.356263
2      precip    0.138635
3  is_weekend    0.137000

RMSE for House 6.0: 144.30

Finally going to look at the hyperparameters for the Decision tree

In [92]:
df_model = final_df.drop('House', axis=1)

#combining each house into one and averaging out the values
df_model = df_model.groupby('date').agg(
    Aggregate = ('Aggregate', 'mean'),
    Temp = ('temp', 'mean'),
    Humid = ('humidity', 'mean'),
    Precip = ('precip', 'mean'),
    is_weekend = ('is_weekend', 'first')
).reset_index()

df_model['Aggregate'] = df_model['Aggregate'].astype(float)
df_model['Temp'] = df_model['Temp'].astype(float)
df_model['Humid'] = df_model['Humid'].astype(float)
df_model['is_weekend'] = df_model['is_weekend'].astype(int)
df_model['Precip'] = df_model['Precip'].astype(float)
df_model = df_model.drop('date', axis=1)

#training model
y = df_model['Aggregate']
features = ['Temp', 'Humid', 'Precip', 'is_weekend']
X = df_model[features]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#setting paramaters
param_distr = {
    'max_depth': [None] + list(randint(5,50).rvs(10)),
    'min_samples_split': randint(2,20),
    'min_samples_leaf': randint(1,10),
    'max_features': ['sqrt', 'log2']
}

dt_model = DecisionTreeRegressor(random_state=42)

#Start the RandomSearchCV
random_search = RandomizedSearchCV(estimator=dt_model,
                                   param_distributions=param_distr,
                                   n_iter=20,
                                   cv=5,
                                   random_state=42)
random_search.fit(X_train, y_train)

#Print which is the best options
print(f'Best paramaters: {random_search.best_params_}')
print(f'Best RMSE: {random_search.best_score_}')
print(f'Best estimator: {random_search.best_estimator_}')

feature_importance = random_search.best_estimator_.feature_importances_
importance_df = pd.DataFrame({'Feature': features, 'Importance': feature_importance})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

#Display results
print(importance_df)


Best paramaters: {'max_depth': 24, 'max_features': 'sqrt', 'min_samples_leaf': 7, 'min_samples_split': 12}
Best RMSE: 0.28795242164845075
Best estimator: DecisionTreeRegressor(max_depth=24, max_features='sqrt', min_samples_leaf=7,
                      min_samples_split=12, random_state=42)
      Feature  Importance
0        Temp    0.695242
1       Humid    0.201628
3  is_weekend    0.077771
2      Precip    0.025359


Decision Tree preformed slightly worse with 31% accuracy. The important features still the same with Temp at top by wide margin